## Step 1: Build the respondent table
Joins the survey name list to the public register and keeps **only the 200 respondents** with matching names ad in the "survey_listC" dataset.
Then computes age on 1 July 2025, bins it (20-39 / 40-59 / 60+), and checks the profiles
against the released file.

In [ ]:
import unicodedata
import pandas as pd

REGISTER = r"..\dataC\public_data_registerC.xlsx"
SURVEY_LIST = r"..\dataC\survey_listC.txt"
RELEASED = r"..\dataC\anonymised_dataC.csv"   # only used for the sanity check at the end
REF_DATE = pd.Timestamp("2025-07-01")   # age reference date from the report

reg = pd.read_excel(REGISTER)
names = [l.strip() for l in open(SURVEY_LIST, encoding="utf-8") if l.strip()]
released = pd.read_csv(RELEASED)



register: (1527, 7) | survey list: 200 | released: (200, 5)


,name,sex,dob,zip,citizenship,marital_status,last_voted
0,"Julian-Ambrose, Michelle",Female,1998-05-12,2200,Denmark,Never married,1
1,"Ironcloud, Malachi",Male,1999-04-18,2100,Denmark,Never married,1
2,"Thompson, Christopher",Male,2001-12-10,2300,Denmark,Never married,1
3,"Cole, Amos",Male,1998-08-04,2300,Denmark,Never married,1
4,"Gibson, Isaiah",Male,2001-03-15,2100,Denmark,Never married,1


In [2]:
# normalise names and join
def norm(s):
    return " ".join(unicodedata.normalize("NFKC", str(s)).split()).casefold()

reg["key"] = reg["name"].map(norm)
survey = pd.DataFrame({"name_survey": names})
survey["key"] = survey["name_survey"].map(norm)

# sanity checks before joining
assert survey["key"].is_unique, "duplicate names in survey list"
dup_in_reg = reg[reg["key"].duplicated(keep=False) & reg["key"].isin(survey["key"])]
print("survey names that are ambiguous in the register:", len(dup_in_reg))

# only the register rows whose name is on the survey list (other duplicates in the register are irrelevant)
reg_match = reg[reg["key"].isin(survey["key"])]
resp = survey.merge(reg_match, on="key", how="left", validate="one_to_one", indicator=True)
unmatched = resp.loc[resp["_merge"] == "left_only", "name_survey"].tolist()
print("matched:", (resp["_merge"] == "both").sum(), "| unmatched:", unmatched)
assert not unmatched and len(resp) == 200

survey names that are ambiguous in the register: 0
matched: 200 | unmatched: []


In [3]:
# handle date range
d = resp["dob"]
resp["age"] = (REF_DATE.year - d.dt.year
               - ((d.dt.month > REF_DATE.month) |
                  ((d.dt.month == REF_DATE.month) & (d.dt.day > REF_DATE.day))).astype(int))
resp["age_group"] = pd.cut(resp["age"], bins=[0, 39, 59, 200],
                           labels=["20-39", "40-59", "60+"])

# register coding: last_voted 0 = paper, 1 = online, 2 = did not vote
resp["evote"] = resp["last_voted"]
assert resp["evote"].isin([0, 1]).all()   # every respondent voted

# build the respondent table
respondents = resp[["name", "sex", "dob", "age", "age_group", "evote", "zip",
                    "citizenship", "marital_status"]].reset_index(drop=True)
respondents.head(10)

,name,sex,dob,age,age_group,evote,zip,citizenship,marital_status
0,"Mcnally, Kyrie",Female,1975-04-27,50,40-59,0,2200,Denmark,Never married
1,"Meraz, Krista",Female,1980-04-29,45,40-59,0,2400,Denmark,Married/separated
2,"al-Rizk, Zainuddeen",Male,1985-10-26,39,20-39,0,2300,Denmark,Married/separated
3,"Powers, Hannah",Female,1959-03-11,66,60+,0,2400,Denmark,Widowed
4,"Thompson, Sarah",Female,1999-04-18,26,20-39,0,2200,Denmark,Never married
5,"Buccieri, Nicholas",Male,1968-06-09,57,40-59,0,2300,Denmark,Never married
6,"al-Abdelrahman, Thaamira",Female,1982-08-21,42,40-59,0,2300,Poland,Married/separated
7,"Lee, Savonh",Male,1949-01-23,76,60+,0,2300,Denmark,Married/separated
8,"Reeves, J'Quan",Male,1960-07-22,64,60+,0,2400,Germany,Never married
9,"Johnson, Nicholas",Male,1959-05-24,66,60+,0,2300,Macedonia,Married/separated


In [4]:
keys = ["sex", "age_group", "evote"]
a = respondents.groupby(keys, observed=True).size().rename("respondents")
b = released.groupby(keys).size().rename("released")
check = pd.concat([a, b], axis=1)
check["match"] = check["respondents"] == check["released"]
display(check)
print("All 12 profiles match:", check["match"].all(), "| min profile size:", check[["respondents","released"]].min().min())

respondents  released  match
sex    age_group evote                              
Female 20-39     0               17        17   True
                 1                9         9   True
       40-59     0               34        34   True
                 1                7         7   True
       60+       0               20        20   True
                 1                5         5   True
Male   20-39     0               20        20   True
                 1               12        12   True
       40-59     0               30        30   True
                 1               15        15   True
       60+       0               25        25   True
                 1                6         6   True

All 12 profiles match: True | min profile size: 5


In [5]:
respondents.to_csv(r"../dataC/respondents_tableC.csv", index=False)